In [29]:
import tensorflow as tf
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt

import copy
import random
import math

from sqlalchemy import create_engine
from tensorflow.keras import layers
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import CSVLogger
from sklearn.model_selection import train_test_split

from datetime import datetime

In [2]:
dbConnectionString = "sqlite:///C:/Users/graye/dev/python/fantasyBaseballPlayerData/updated_database/baseball_info.db"

engine = create_engine(dbConnectionString)

In [273]:
timeSeriesHittingQuery = "SELECT SeasonStatsHitting.playerId,season,Bios.dob,plateAppearances,atBats,runs,hits,singles,doubles,triples,homeRuns,rbis,sacHits,sacFlies,hitByPitch,walks,intentionalWalks,strikeOuts,stolenBases,caughtStealing,groundedIntoDoublePlays,catcherInterference,battingAverage,onBasePercentage,sluggingPercentage,onBasePlusSlugging,runsPerTob,rbisPerBip FROM SeasonStatsHitting INNER JOIN Bios on Bios.playerId = SeasonStatsHitting.playerId WHERE season BETWEEN 2018 and 2025 and season != 2020 ORDER BY SeasonStatsHitting.playerId,season"

df = pd.read_sql(timeSeriesHittingQuery, engine)

In [274]:
def calculate_age_from_timestamps(birth_timestamp, current_timestamp):
    birth_date   = datetime.fromtimestamp(birth_timestamp)
    current_date = datetime.fromtimestamp(current_timestamp)

    age = current_date.year - birth_date.year

    # Adjust age if the birthday hasn't occurred yet in the current year
    if (current_date.month, current_date.day) < (birth_date.month, birth_date.day):
        age -= 1
    
    return age

In [275]:
twentyEighteenDateString    = "2018-03-29 00:00:00"
twentyNineteenDateString    = "2019-03-20 00:00:00"
twentyTwentyOneDateString   = "2021-04-01 00:00:00"
twentyTwentyTwoDateString   = "2022-04-07 00:00:00"
twentyTwentyThreeDateString = "2023-03-30 00:00:00"
twentyTwentyFourDateString  = "2024-03-20 00:00:00"
twentyTwentyFiveDateString  = "2025-03-18 00:00:00"

# Parse the string into a datetime object
twentyEighteenDateString    = datetime.strptime(twentyEighteenDateString    , "%Y-%m-%d %H:%M:%S")
twentyNineteenDateString    = datetime.strptime(twentyNineteenDateString    , "%Y-%m-%d %H:%M:%S")
twentyTwentyOneDateString   = datetime.strptime(twentyTwentyOneDateString   , "%Y-%m-%d %H:%M:%S")
twentyTwentyTwoDateString   = datetime.strptime(twentyTwentyTwoDateString   , "%Y-%m-%d %H:%M:%S")
twentyTwentyThreeDateString = datetime.strptime(twentyTwentyThreeDateString , "%Y-%m-%d %H:%M:%S")
twentyTwentyFourDateString  = datetime.strptime(twentyTwentyFourDateString  , "%Y-%m-%d %H:%M:%S")
twentyTwentyFiveDateString  = datetime.strptime(twentyTwentyFiveDateString  , "%Y-%m-%d %H:%M:%S")

# Convert to Unix timestamp
twentyEighteenTimeStamp    = int(twentyEighteenDateString    .timestamp())
twentyNineteenTimeStamp    = int(twentyNineteenDateString    .timestamp())
twentyTwentyOneTimeStamp   = int(twentyTwentyOneDateString   .timestamp())
twentyTwentyTwoTimeStamp   = int(twentyTwentyTwoDateString   .timestamp())
twentyTwentyThreeTimeStamp = int(twentyTwentyThreeDateString .timestamp())
twentyTwentyFourTimeStamp  = int(twentyTwentyFourDateString  .timestamp())
twentyTwentyFiveTimeStamp  = int(twentyTwentyFiveDateString  .timestamp())

seasonToStartTimeStamp = {}

seasonToStartTimeStamp[2018] = twentyEighteenTimeStamp   
seasonToStartTimeStamp[2019] = twentyNineteenTimeStamp   
seasonToStartTimeStamp[2021] = twentyTwentyOneTimeStamp  
seasonToStartTimeStamp[2022] = twentyTwentyTwoTimeStamp  
seasonToStartTimeStamp[2023] = twentyTwentyThreeTimeStamp
seasonToStartTimeStamp[2024] = twentyTwentyFourTimeStamp 
seasonToStartTimeStamp[2025] = twentyTwentyFiveTimeStamp

In [276]:
#drop players that cannot create a valid window (ie have not played multiple season)
hasMultipleSeasons = df["playerId"].value_counts() > 1
df = df[df["playerId"].isin(hasMultipleSeasons[hasMultipleSeasons].index)]

#drop players with no plater appearances
df.drop(df[df['plateAppearances'] == 0].index, inplace=True)

#get player ages
df['dob'] = df.apply(lambda row: calculate_age_from_timestamps(row['dob'], seasonToStartTimeStamp[row['season']]), axis=1)

#log of all numerical columns to smooth distribution
logColumns     = df.columns[3:]
df[logColumns] = np.log(df[logColumns] + 1)

In [277]:
def getWindowedFeaturesAndLabels(frame, maxWindowSize):
    frameArray = frame.values.tolist()

    inputs = []
    labels = []
    
    r = 0

    curWindow = []

    playerId = frameArray[0][0]
    
    while r < len(frameArray):
        curPlayerId = frameArray[r][0]

        if curPlayerId != playerId:
            window = []
            
            windowR = 0

            while windowR <= len(curWindow):
                if len(window) == maxWindowSize or windowR == len(curWindow):
                    label = window.pop()

                    labelTest = copy.deepcopy(label)

                    relevantLabels = labelTest[5:6] + labelTest[10:12] + labelTest[18:19] + labelTest[25:26]

                    if len(window) < maxWindowSize - 1:
                        diff = maxWindowSize - 1 - len(window)

                        while diff > 0:
                            padding = [0] * 28
                            window.append(padding)

                            diff -= 1

                    windowArray = [inner_list[2:] for inner_list in window]

                    inputs.append(copy.deepcopy(windowArray))
                    labels.append(copy.deepcopy(relevantLabels))

                    window.append(label)
                    window.pop(0)

                if windowR < len(curWindow):
                    window.append(curWindow[windowR])
                
                windowR += 1

            curWindow = []
            playerId  = curPlayerId

        curWindow.append(frameArray[r])

        r += 1

    return inputs,labels

In [278]:
fullFeatures,fullLabels = getWindowedFeaturesAndLabels(df, 4)

In [279]:
fullFeaturesArray = np.array(fullFeatures)
fullLabelsArray   = np.array(fullLabels)

In [280]:
runBins              = np.linspace(min(fullLabelsArray[:,0]), max(fullLabelsArray[:,0]), num=4)
homeRunsBins         = np.linspace(min(fullLabelsArray[:,1]), max(fullLabelsArray[:,1]), num=4)
rbiBins              = np.linspace(min(fullLabelsArray[:,2]), max(fullLabelsArray[:,2]), num=4)
stolenBaseBins       = np.linspace(min(fullLabelsArray[:,3]), max(fullLabelsArray[:,3]), num=4)
onBasePercentageBins = np.linspace(min(fullLabelsArray[:,4]), max(fullLabelsArray[:,4]), num=4)

In [281]:
runsBinned             = np.digitize(fullLabelsArray[:,0], runBins, right=True)
homeRunsBinned         = np.digitize(fullLabelsArray[:,1], homeRunsBins, right=True)
rbisBinned             = np.digitize(fullLabelsArray[:,2], rbiBins, right=True)
stolenBasesBinned      = np.digitize(fullLabelsArray[:,3], stolenBaseBins, right=True)
onBasePercentageBinned = np.digitize(fullLabelsArray[:,4], onBasePercentageBins, right=True)

In [282]:
fullLabelsArrayBinned = np.column_stack((runsBinned, homeRunsBinned, rbisBinned, stolenBasesBinned, onBasePercentageBinned))

In [283]:
#remove unique rows from bin, features and labels and split them evenly between data sets

In [284]:
uniqueRowsIndexes = binnedDf.drop_duplicates(keep=False).index

uniqueRowsIndexesArray = np.array(uniqueRowsIndexes)
fullLabelsArrayBinnedFiltered = np.delete(fullLabelsArrayBinned, uniqueRowsIndexesArray, axis=0)
fullFeaturesArrayFiltered = np.delete(fullFeaturesArray, uniqueRowsIndexesArray, axis=0)
fullLabelsArrayFiltered = np.delete(fullLabelsArray, uniqueRowsIndexesArray, axis=0)

In [285]:
trainFeatures,valTestFeatures,trainLabels,valTestLables = train_test_split(fullFeaturesArrayFiltered, fullLabelsArrayFiltered, stratify=fullLabelsArrayBinnedFiltered, test_size=0.2)

In [286]:
valFeatures,testFeatures,valLabels,testLabels = train_test_split(valTestFeatures, valTestLables, test_size=0.5)

In [289]:
normalizeInput = layers.Normalization()
normalizeInput.adapt(trainFeatures)

In [310]:
#l2/l1 regularization, dropout, activation function, optmizier, loss function, hidden units, hidden layers, batch size, batch normalization




def get_model():
    model = tf.keras.Sequential([
        layers.Input(shape=(3, 26)),
        layers.Masking(mask_value=0., input_shape=(3, 26)),
        normalize,
        layers.LSTM(units=20, dropout=0.2, recurrent_dropout=0.2, kernel_regularizer=regularizers.l2(0.001), return_sequences=True),
        layers.LayerNormalization(),
        layers.Dense(units = 5)
    ])

    return model

In [311]:
curModel = get_model()

curModel.compile(optimizer = tf.keras.optimizers.Adam(weight_decay=0.0001), loss="mse", metrics=[tf.keras.metrics.R2Score()])

curModel.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┓
┃ Layer (type)                        ┃ Output Shape               ┃        Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━┩
│ masking_5 (Masking)                 │ (None, 3, 26)              │              0 │
├─────────────────────────────────────┼────────────────────────────┼────────────────┤
│ normalization (Normalization)       │ (None, 3, 26)              │             53 │
├─────────────────────────────────────┼────────────────────────────┼────────────────┤
│ lstm_6 (LSTM)                       │ (None, 3, 20)              │          3,760 │
├─────────────────────────────────────┼────────────────────────────┼────────────────┤
│ layer_normalization_6               │ (None, 3, 20)              │             40 │
│ (LayerNormalization)                │                            │                │
├─────────────────────────────────────┼────────────────────────────┼────────────────┤
│ lstm_7 (LSTM)                       │ (None, 20)                 │          3,280 │
├─────────────────────────────────────┼────────────────────────────┼────────────────┤
│ layer_normalization_7               │ (None, 20)                 │             40 │
│ (LayerNormalization)                │                            │                │
├─────────────────────────────────────┼────────────────────────────┼────────────────┤
│ dense_5 (Dense)                     │ (None, 5)                  │            105 │
└─────────────────────────────────────┴────────────────────────────┴────────────────┘

 Total params: 7,278 (28.43 KB)

 Trainable params: 7,225 (28.22 KB)

 Non-trainable params: 53 (216.00 B)

In [312]:
curModel.fit(trainFeatures, trainLabels, batch_size=32, epochs=50, validation_data=(valFeatures, valLabels))

Epoch 1/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 12s 34ms/step - loss: 2.0699 - r2_score: -3.0859 - val_loss: 0.9215 - val_r2_score: -0.2608
Epoch 2/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.9880 - r2_score: -0.5466 - val_loss: 0.8335 - val_r2_score: 0.1972
Epoch 3/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.8800 - r2_score: -0.0545 - val_loss: 0.7756 - val_r2_score: 0.3716
Epoch 4/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.8288 - r2_score: 0.1546 - val_loss: 0.7247 - val_r2_score: 0.4372
Epoch 5/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 0.7788 - r2_score: 0.2404 - val_loss: 0.6871 - val_r2_score: 0.4834
Epoch 6/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.7438 - r2_score: 0.3186 - val_loss: 0.6581 - val_r2_score: 0.4913
Epoch 7/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - loss: 0.7117 - r2_score: 0.3649 - val_loss: 0.6473 - val_r2_score: 0.5251
Epoch 8/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.6913 - r2_score: 0.4098 - val_loss: 0.6242 - 

In [ ]:
loss: 0.4811 - r2_score: 0.6423 - val_loss: 0.5177 - val_r2_score: 0.6127

In [299]:
baselineResults = curModel.evaluate(testFeatures, testLabels)

8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.5561 - r2_score: 0.6074 


In [300]:
for featureIndex in range(testFeatures.shape[2]):
   print(train_df.columns[featureIndex+2])
   shuffledTestFeatures = testFeatures.copy()

   testColumnElements = [shuffledTestFeatures[i][j][featureIndex] for i in range(len(shuffledTestFeatures)) for j in range(len(shuffledTestFeatures[i]))]

   random.shuffle(testColumnElements)
   
   k = 0
   
   for i in range(len(shuffledTestFeatures)):
       for j in range(len(shuffledTestFeatures[i])):
           shuffledTestFeatures[i][j][featureIndex] = testColumnElements[k]
   
           k += 1

   shuffledResults = curModel.evaluate(shuffledTestFeatures, testLabels)

   #print(shuffledResults[0] / baselineResults[0])
   print(shuffledResults[1] / baselineResults[1])

dob
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.9398 - r2_score: 0.3614 
0.5950080490806503
plateAppearances
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.8694 - r2_score: 0.4205 
0.6922963929953079
atBats
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.8359 - r2_score: 0.4422 
0.7279085950993531
runs
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.8857 - r2_score: 0.3757 
0.6185339722620222
hits
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.9059 - r2_score: 0.3940 
0.6485906602404853
singles
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.8987 - r2_score: 0.4014 
0.6607752099698309
doubles
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.9004 - r2_score: 0.4143
0.682048165745711
triples
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.7602 - r2_score: 0.4936 
0.8125608243797164
homeRuns
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.9223 - r2_score: 0.3630 
0.5976648442493573
rbis
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - loss: 0.8610 - r2_score: 0.4212
0.6933840442749479
sacHits
8/8 ━━━

In [ ]:
# import numpy as np
# from sklearn.model_selection import train_test_split

# # Sample data and target variable
# X = np.array([[1, 2], [3, 4], [5, 6], [7, 8], [9, 10], [11, 12]])
# y = np.array([0, 1, 0, 1, 0, 1])  # Target variable for stratification

# # Get original indices
# original_indices = np.arange(len(X))

# # Perform stratified train-test split on indices
# # train_indices and test_indices will contain the indices for the stratified split
# train_indices, test_indices, _, _ = train_test_split(
#     original_indices, y, test_size=0.3, random_state=42, stratify=y
# )

# print("Train indices:", train_indices)
# print("Test indices:", test_indices)

# # You can then use these indices to select your data:
# X_train = X[train_indices]
# y_train = y[train_indices]
# X_test = X[test_indices]
# y_test = y[test_indices]

# print("\nX_train:\n", X_train)
# print("y_train:", y_train)
# print("\nX_test:\n", X_test)
# print("y_test:", y_test)

In [ ]:
binnedDf = pd.DataFrame({
    'runs': runsBinned,
    'homeRuns': homeRunsBinned,
    'rbis': rbisBinned,
    'stolenBases': stolenBasesBinned,
    'onBasePercentage': onBasePercentageBinned
})